# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [ ]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml
import os
os.environ["UNSLOTH_DISABLE_PATCHING"] = "1"
%pip install transformers==4.56.2

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu128
CUDA available: True


In [ ]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    "model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    "max_seq_length": 1024,
    "lora_r": 16,
    "lora_alpha": 32,
    "learning_rate": 1e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 2,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 50,
    "eval_steps": 1000,
    "save_steps": 1000,
    "max_train_samples_per_source": 12000,
    "eval_size": 0.1,
    "output_dir": "/content/output",
}

CONFIG

{'model_name': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
 'max_seq_length': 1024,
 'lora_r': 16,
 'lora_alpha': 32,
 'learning_rate': 0.0001,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 2,
 'gradient_accumulation_steps': 2,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 50,
 'eval_steps': 1000,
 'save_steps': 1000,
 'max_train_samples_per_source': 12000,
 'eval_size': 0.1,
 'output_dir': '/content/output'}

In [ ]:
# Data catalog using the resources listed in contest_docs/03_Data_Design.md.
BASE_PATH = "/content/drive/MyDrive/svg_project"


TRAIN_CSV_PATH = f"{BASE_PATH}/train.csv"

In [ ]:
import pandas as pd
from datasets import Dataset

def load_competition_train_csv(csv_path, max_samples=None):
    df = pd.read_csv(csv_path)

    # keep only needed columns
    df = df[["prompt", "svg"]].copy()

    # basic cleanup
    df["prompt"] = df["prompt"].astype(str).str.strip()
    df["svg"] = df["svg"].astype(str).str.strip()

    # keep only valid-looking rows
    df = df[(df["prompt"] != "") & (df["svg"] != "")]
    df = df[df["svg"].str.contains("<svg", case=False, na=False)]

    if max_samples is not None and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=SEED).reset_index(drop=True)

    print("Loaded rows:", len(df))
    print(df.iloc[0])

    return Dataset.from_pandas(df, preserve_index=False)

In [ ]:
train_raw = load_competition_train_csv(
    TRAIN_CSV_PATH,
    CONFIG["max_train_samples_per_source"],
)

train_raw = train_raw.shuffle(seed=SEED)
print("Total usable rows:", len(train_raw))

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

Loaded rows: 12000
prompt    A black icon featuring a vertical rectangle wi...
svg       <svg xmlns="http://www.w3.org/2000/svg" viewBo...
Name: 0, dtype: object
Total usable rows: 12000
Train rows: 10800
Eval rows: 1200


{'prompt': 'The image features two overlapping rectangular shapes, one orange and one yellow, with the orange rectangle positioned in front and featuring a white exclamation mark and dot.',
 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#FFC646" fill-opacity="1.0"  filling="0" d="M106.9769515991211 56.07851791381836 L146.20391845703125 52.18378829956055 C152.552734375 51.55351638793945 152.552734375 51.55351638793945 153.18319702148438 57.90234375 L166.2068328857422 189.07733154296875 C166.83709716796875 195.42616271972656 166.83709716796875 195.42616271972656 160.48828125 196.05645751953125 L121.2611312866211 199.951171875 C114.91230773925781 200.58145141601562 114.91230773925781 200.58145141601562 114.28203582763672 194.2326202392578 L101.2583999633789 63.057621002197266 C100.62812805175781 56.70878982543945 100.62812805175781 56.70878982543945 106.9769515991211 56.07851791381836 Z"></path>\n<path fill="#FF5

In [ ]:
SYSTEM_PROMPT = (
    "Generate only valid SVG code. "
    "Return exactly one complete <svg>...</svg> element. "
    "Do not include markdown, comments, explanations, or extra text."
)

def format_sft_text(example):
    text = (
        "Generate SVG for the following description.\n"
        f"Description: {example['prompt']}\n"
        "SVG:\n"
        f"{example['svg']}"
    )
    return {"text": text}

train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

Map:   0%|          | 0/10800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generate SVG for the following description.
Description: The image features two overlapping rectangular shapes, one orange and one yellow, with the orange rectangle positioned in front and featuring a white exclamation mark and dot.
SVG:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#FFC646" fill-opacity="1.0"  filling="0" d="M10


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-1.5b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.17 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
from trl.trainer.sft_trainer import SFTTrainer
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=training_args,
)

print(type(trainer))
train_result = trainer.train()
train_result

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/10800 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=12):   0%|          | 0/1200 [00:00<?, ? examples/s]

<class 'UnslothSFTTrainer.UnslothSFTTrainer'>


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,800 | Num Epochs = 1 | Total steps = 2,700
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss
1000,0.433000,0.432238
2000,0.427200,0.397163


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=2700, training_loss=0.44689595116509334, metrics={'train_runtime': 6993.5763, 'train_samples_per_second': 1.544, 'train_steps_per_second': 0.386, 'total_flos': 8.630713698736742e+16, 'train_loss': 0.44689595116509334, 'epoch': 1.0})

In [ ]:
import os

SAVE_PATH = "/content/drive/MyDrive/svg_project/final_model"
os.makedirs(SAVE_PATH, exist_ok=True)

trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Saved model to:", SAVE_PATH)
print(os.listdir(SAVE_PATH))


save_path = "/content/output"

os.makedirs(save_path, exist_ok=True)

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Saved to:", save_path)

os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

Saved model to: /content/drive/MyDrive/svg_project/final_model
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json']
Saved to: /content/output
Saved adapter + tokenizer to: /content/output


In [ ]:
!ls
print(CONFIG["output_dir"])

drive			      output	   unsloth_compiled_cache
huggingface_tokenizers_cache  sample_data
/content/output
